# 13.2 · 变分自编码器 / Variational Autoencoder (VAE)

> **课程定位 / Where this fits**
> 第 2 课，**Part 13 · 生成模型**。第一个真正能"生成"的模型。
> Lesson 2, **Part 13 · Generative Models**. The first model that truly "generates."
>
> 13.1 发现普通 AE 不能可靠生成——隐空间有"空洞"。**变分自编码器(VAE)** 用一个优雅的改造解决它：**编码器不再把图压成一个点, 而是压成一个概率分布(均值+方差)**, 并用一项损失**强迫整个隐空间逼近标准正态分布 N(0,1)**。训练好后, 只要从 N(0,1) **随便采一个点, 解码就能得到一个全新的、像样的数字**！本课讲清 **ELBO** 损失、**重参数化技巧**(让"采样"也能反向传播), 从零搭 VAE, 并**亲手采样生成数字 + 隐空间插值**。
> Lesson 13.1 showed a plain AE can't reliably generate — its latent has "holes." The **VAE** fixes this elegantly: **the encoder maps an image to a probability distribution (mean + variance), not a point**, and a loss term **forces the whole latent space toward a standard normal N(0,1)**. After training, just **sample any point from N(0,1), decode, and get a new, plausible digit!** We cover the **ELBO** loss, the **reparameterization trick** (so "sampling" is backprop-able), build a VAE from scratch, and **generate digits + interpolate the latent space**.
>
> 💼 **实战/面试视角**："VAE 和 AE 的区别 / ELBO 两项 / 重参数化为什么 / KL 项的作用 / VAE vs GAN" 是生成模型必考。
> 💼 **Practical/interview angle:** "VAE vs AE / the two ELBO terms / why reparameterization / role of KL / VAE vs GAN" — generative essentials.

> 📐 **符号约定 / Notation**
> - $q(z\mid x)$ —— 编码器输出的隐变量分布(近似后验) / encoder's latent distribution
> - $\mu, \sigma$ —— 该分布的均值、标准差 / its mean and std
> - ELBO —— 证据下界(VAE 的优化目标) / Evidence Lower Bound

> 💡 **面试相关 / Interview-relevant**
> - "VAE 相比 AE 的关键改造"（出镜率 ★★★★★）
> - "ELBO = 重建项 + KL 项, 各管什么"（★★★★★）
> - "重参数化技巧解决什么问题"（★★★★★）
> - "VAE 生成图为什么偏模糊"（★★★★）
> - "VAE vs GAN vs Diffusion"（★★★★）

---

## 学习目标 / Learning Objectives
1. 理解 VAE 如何把隐空间约束成 N(0,1) 从而能生成。
   Understand how the VAE constrains the latent to N(0,1) to enable generation.
2. 掌握 **ELBO**(重建 + KL)与**重参数化技巧**。
   Master the ELBO (reconstruction + KL) and the reparameterization trick.
3. **从零搭 VAE** 训练并**采样生成数字**。
   Build a VAE from scratch, train, and generate digits by sampling.
4. 做**隐空间插值**, 看数字平滑过渡。
   Interpolate the latent and watch digits morph smoothly.

## 目录 / TOC
1. [从 AE 到 VAE：编码成分布 ⭐](#1)
2. [ELBO 与重参数化技巧 ⭐](#2)
3. [搭建并训练 VAE ⭐](#3)
4. [采样生成 + 隐空间插值 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 从 AE 到 VAE：编码成分布 ⭐ / From AE to VAE: Encode to a Distribution

普通 AE 把图编码成隐空间里的**一个点**, 隐空间杂乱有空洞→不能采样。VAE 的两个关键改造：
A plain AE encodes an image to **a single point**; the latent is messy with holes → can't sample. The VAE's two key changes:
1. **编码成一个分布, 而不是一个点**：编码器对每张图输出一个**高斯分布的参数**(均值 $\mu$ 和方差 $\sigma^2$), 隐变量 $z$ 从这个分布里**采样**得到。这让相近的输入映射到隐空间里**重叠的区域**, 填补空洞。
   **Encode to a distribution, not a point:** the encoder outputs **Gaussian parameters** ($\mu, \sigma^2$) per image; the latent $z$ is **sampled** from it. Similar inputs map to **overlapping regions**, filling the holes.
2. **约束这个分布接近 N(0,1)**：用一项 **KL 散度损失**, 逼每张图的隐分布 $q(z\mid x)$ 都靠近**标准正态 N(0,1)**。于是整个隐空间被"整理"成一个规整的、连续的、以原点为中心的分布。
   **Constrain that distribution toward N(0,1):** a **KL-divergence loss** pushes each image's $q(z\mid x)$ toward the **standard normal N(0,1)**. The whole latent gets "organized" into a smooth, continuous, origin-centered distribution.

**生成就水到渠成**：既然训练后整个隐空间≈N(0,1), 那**从 N(0,1) 随便采一个 $z$, 它一定落在"有意义"的区域, 解码就得到一个新数字**——不再有空洞问题！
**Generation follows naturally:** since the latent ≈ N(0,1) after training, **sampling any $z$ from N(0,1) lands in a "meaningful" region, and decoding yields a new digit** — no more holes!


<a id="2"></a>
## 2. ELBO 与重参数化技巧 ⭐ / ELBO & Reparameterization

VAE 的损失叫 **ELBO(证据下界)**, 由**两项**组成(面试必背)：
The VAE loss is the **ELBO (Evidence Lower Bound)**, with **two terms** (must-know):

$$\mathcal{L} = \underbrace{\text{重建误差}}_{\text{让}\hat{x}\approx x} + \underbrace{\beta \cdot \text{KL}\big(q(z\mid x)\,\|\,N(0,1)\big)}_{\text{让隐分布靠近}N(0,1)}$$

- **重建项**：解码出的图要像输入(和 AE 一样, 保证有用)。
  **Reconstruction term:** the decoded image should match the input (like AE; keeps it useful).
- **KL 项**：把每张图的隐分布拉向 N(0,1)(让隐空间规整、可采样)。这是 VAE 区别于 AE 的灵魂。
  **KL term:** pulls each image's latent distribution toward N(0,1) (organizes the latent for sampling). The soul of the VAE vs AE.
- 两者**互相拉扯**：重建想"为每张图留独特编码", KL 想"都挤成 N(0,1)"——平衡出一个既能重建又规整的隐空间。
  They **tug against each other:** reconstruction wants distinct codes per image, KL wants everything as N(0,1) — the balance yields a latent that both reconstructs and is well-organized.

**重参数化技巧(reparameterization trick)**(面试高频)：训练要反向传播, 但"从分布里采样 $z$"这个操作是**随机的、不可导**的, 梯度过不去。技巧：把随机性"挪到外面"——不直接采 $z$, 而是写成 $z = \mu + \sigma \odot \epsilon$, 其中 $\epsilon \sim N(0,1)$ 是外部噪声。这样 $\mu, \sigma$ 是确定的(可导), 随机性只在 $\epsilon$ 里(不需要对它求导), **梯度就能顺利流过 $\mu, \sigma$**。
**Reparameterization trick** (high-frequency): training needs backprop, but "sampling $z$ from a distribution" is **random and non-differentiable** — gradients can't pass. The trick: move the randomness "outside" — instead of sampling $z$ directly, write $z = \mu + \sigma \odot \epsilon$ where $\epsilon \sim N(0,1)$ is external noise. Now $\mu, \sigma$ are deterministic (differentiable) and randomness lives only in $\epsilon$ (no gradient needed there), so **gradients flow through $\mu, \sigma$**.


<a id="3"></a>
## 3. 搭建并训练 VAE ⭐ / Build & Train the VAE

从零搭 VAE：编码器输出 $\mu$ 和 $\log\sigma^2$(用 log 方差保证正且数值稳定), 重参数化采样 $z$, 解码重建。损失 = 重建(BCE) + KL。
Build the VAE from scratch: the encoder outputs $\mu$ and $\log\sigma^2$ (log-variance for positivity and stability), reparameterize to sample $z$, decode. Loss = reconstruction (BCE) + KL.


In [ ]:
import os, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
sns.set_theme(style="white"); torch.manual_seed(0)

DATA_ROOT = os.path.expanduser("~/.cache/dsfs_cv")
train_full = datasets.MNIST(DATA_ROOT, train=True, download=True, transform=transforms.ToTensor())
test_full  = datasets.MNIST(DATA_ROOT, train=False, download=True, transform=transforms.ToTensor())
train_loader = DataLoader(Subset(train_full, range(15000)), batch_size=128, shuffle=True)
LATENT = 16

class VAE(nn.Module):
    def __init__(self, latent=LATENT):
        super().__init__()
        self.enc = nn.Sequential(nn.Flatten(), nn.Linear(784,400), nn.ReLU())
        self.fc_mu = nn.Linear(400, latent)               # 输出均值 μ / mean
        self.fc_logvar = nn.Linear(400, latent)           # 输出 log方差 / log-variance
        self.dec = nn.Sequential(nn.Linear(latent,400), nn.ReLU(), nn.Linear(400,784), nn.Sigmoid())
    def encode(self, x):
        h = self.enc(x); return self.fc_mu(h), self.fc_logvar(h)
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5*logvar)                       # σ = exp(0.5·logσ²) / std from log-variance
        eps = torch.randn_like(std)                       # 外部噪声 ε~N(0,1) / external noise
        return mu + std*eps                               # 重参数化: z=μ+σ·ε(随机性在ε里, 梯度可过μ,σ) / reparameterize
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.dec(z).view(-1,1,28,28), mu, logvar

def vae_loss(xhat, x, mu, logvar):
    recon = F.binary_cross_entropy(xhat, x, reduction="sum")          # 重建项(BCE) / reconstruction
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())      # KL(q||N(0,1)) 的解析式 / KL term
    return recon + kl

torch.manual_seed(0); vae = VAE(); opt = torch.optim.Adam(vae.parameters(), 1e-3)
for epoch in range(20):
    tot = 0
    for xb, _ in train_loader:
        xhat, mu, logvar = vae(xb); loss = vae_loss(xhat, xb, mu, logvar)
        opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()
print(f"VAE 训练完成, 最终每张图损失 ≈ {tot/len(train_loader.dataset):.1f} (重建+KL)")
# 重建效果 / reconstructions
vae.eval(); imgs = torch.stack([test_full[i][0] for i in range(10)])
with torch.no_grad(): recon,_,_ = vae(imgs)
fig, axes = plt.subplots(2,10,figsize=(12,2.6))
for i in range(10):
    axes[0,i].imshow(imgs[i,0],cmap="gray"); axes[0,i].axis("off")
    axes[1,i].imshow(recon[i,0],cmap="gray"); axes[1,i].axis("off")
fig.suptitle("VAE 重建(上=原图, 下=重建): 可辨认但偏'平滑模糊'(VAE 的典型特征)"); plt.tight_layout(); plt.show()
print("重建可辨认; VAE 重建/生成图偏模糊——因为它建模分布+用均值, 倾向'平均化'(对比GAN更锐利)")


<a id="4"></a>
## 4. 采样生成 + 隐空间插值 + 小结 ⭐ / Sampling & Interpolation

**真正的生成**：训练后隐空间≈N(0,1), 所以**直接从 N(0,1) 采样 $z$ → 解码 → 全新数字**(不需要任何输入图)。对比 13.1 普通 AE 随机采样得到的"四不像", VAE 采样应该得到像样的数字。
**True generation:** after training the latent ≈ N(0,1), so **sample $z$ from N(0,1) → decode → a brand-new digit** (no input image needed). Unlike the AE's garbage in 13.1, VAE samples should look like real digits.


In [ ]:
# 从 N(0,1) 采样生成全新数字(无需任何输入图) / generate by sampling N(0,1)
with torch.no_grad():
    z = torch.randn(20, LATENT)                           # 直接从标准正态采样 / sample standard normal
    gen = vae.dec(z).view(-1,1,28,28)
fig, axes = plt.subplots(2,10,figsize=(12,2.6))
for i in range(20): axes[i//10,i%10].imshow(gen[i,0],cmap="gray"); axes[i//10,i%10].axis("off")
fig.suptitle("VAE 生成: 从 N(0,1) 随便采样→解码→全新数字(对比13.1普通AE的'四不像')"); plt.tight_layout(); plt.show()
print("VAE 能可靠生成像样数字(虽偏模糊): 因为隐空间被KL约束成N(0,1), 随便采都落在有意义区域")


In [ ]:
# 隐空间插值: 在两个数字的编码之间走直线, 看平滑过渡 / latent interpolation between two digits
with torch.no_grad():
    a, b = test_full[1][0], test_full[9][0]               # 取两张数字 / two digits
    mu_a,_ = vae.encode(a.unsqueeze(0)); mu_b,_ = vae.encode(b.unsqueeze(0))
    steps = torch.linspace(0,1,10)
    interp = torch.stack([vae.dec((1-t)*mu_a + t*mu_b).view(1,28,28) for t in steps])
fig, axes = plt.subplots(1,10,figsize=(12,1.5))
for i in range(10): axes[i].imshow(interp[i,0],cmap="gray"); axes[i].axis("off")
fig.suptitle("隐空间插值: 从一个数字平滑'变形'到另一个 → 隐空间连续且有意义"); plt.tight_layout(); plt.show()
print("两个数字编码之间的直线, 解码出平滑的中间形态 → VAE 隐空间是连续的(普通AE做不到)")
print("这证明VAE学到了有结构的、可遍历的隐空间, 而非一堆孤立的点")


```
VAE 关键改造: ①编码成分布(μ,σ)而非点 ②KL项强迫隐分布≈N(0,1) → 隐空间规整可采样
ELBO = 重建项(x̂≈x, 保证有用) + KL项(q(z|x)≈N(0,1), 规整隐空间); 两项互相拉扯求平衡
重参数化技巧: 采样不可导→写成 z=μ+σ·ε (ε~N(0,1)), 随机性挪到ε, 梯度可流过μ,σ
生成: 训练后隐空间≈N(0,1) → 从N(0,1)采样z→解码→新数字(无需输入); 对比AE随机采样=垃圾
插值: 两编码间直线解码→平滑过渡 → 隐空间连续有意义
VAE特点: 生成偏模糊(建模分布+均值化); 训练稳定; 有精确隐空间; 对比GAN(锐利但难训)
```

### 💡 面试速查 / Interview cheat-sheet
1. **VAE vs AE**: 编码成分布(μ,σ)+KL约束隐空间≈N(0,1)→能采样生成。
   VAE vs AE: encode to a distribution + KL to N(0,1) → can sample/generate.
2. **ELBO**: 重建项 + KL项; 重建保有用, KL规整隐空间。
   ELBO: reconstruction + KL; reconstruction keeps it useful, KL organizes the latent.
3. **重参数化**: z=μ+σ·ε 让采样可导, 梯度能反传。
   Reparameterization: z=μ+σ·ε makes sampling differentiable.
4. **生成模糊**: VAE 建模分布+均值化→偏模糊(GAN更锐利)。
   Blurry: VAE models distributions/averages → blurry (GAN sharper).
5. **优点**: 训练稳定 + 有结构连续的隐空间(可插值)。
   Pros: stable training + structured, continuous latent (interpolatable).

### 下一节 / Next
**13.3 GAN 基础**——另一条生成路线, 思路完全不同: 让两个网络**对抗博弈**——生成器努力造假, 判别器努力识破, 在博弈中生成器学会造出以假乱真的图。GAN 生成的图比 VAE **锐利**得多, 但训练**出了名地不稳定**(模式坍塌)。我们会从零搭 GAN 并复现这些现象。
**13.3 GAN** — another route, totally different: two networks in an **adversarial game** — a generator fakes, a discriminator detects; through the game the generator learns to produce convincing images. GAN outputs are far **sharper** than VAE, but training is **notoriously unstable** (mode collapse). We'll build a GAN from scratch and reproduce these phenomena.
